# SQL exploratory analysis

Answering concrete questions about sites, payloads and outcomes with SQL.

## Loading the launch dataset into SQLite

The launch table is loaded into a local SQLite database so it can be queried with SQL.

In [ ]:
!pip install jupysql prettytable


### Connecting to the database

Load the SQL extension and open a connection to the SQLite database.

In [ ]:
!pip install jupysql prettytable


In [ ]:
!pip show jupysql

In [38]:
%load_ext sql

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [39]:
import csv, sqlite3
import prettytable
prettytable.DEFAULT = 'DEFAULT'

con = sqlite3.connect('../my_data1.db')
cur = con.cursor()


In [ ]:
!pip install -q pandas

In [40]:
%config SqlMagic.autolimit = None
%config SqlMagic.displaylimit = None

displaylimit: Value None will be treated as 0 (no limit)

In [41]:
%sql sqlite:///../my_data1.db


In [42]:
import pandas as pd
df = pd.read_csv("../data/raw/spacex_launch.csv")
df.to_sql("SPACEXTBL", con, if_exists='replace', index=False,method="multi")


101

Remove rows with a blank date before querying.

In [43]:
#DROP THE TABLE IF EXISTS

%sql DROP TABLE IF EXISTS SPACEXTABLE;

Running query in 'sqlite:///my_data1.db'

++
||
++
++

In [44]:
%sql create table SPACEXTABLE as select * from SPACEXTBL where Date is not null

Running query in 'sqlite:///my_data1.db'

++
||
++
++

## Query tasks

The queries below answer specific questions about launch sites, payload masses and landing outcomes.

In [45]:
%sql select "Launch_Site" from SPACEXTBL group by "Launch_Site" 

Running query in 'sqlite:///my_data1.db'

Launch_Site
CCAFS LC-40
CCAFS SLC-40
KSC LC-39A
VAFB SLC-4E


### Sites starting with 'CCA'

Show five records for launch sites whose name begins with `CCA`.

In [46]:
%sql select * from SPACEXTBL where "Launch_Site" like "CCA%" LIMIT 5

Running query in 'sqlite:///my_data1.db'

Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of Brouere cheese",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Total payload launched for NASA (CRS)

Sum the payload mass for NASA's commercial resupply missions.

In [47]:
%%sql 
select sum(PAYLOAD_MASS__KG_) as "Total_Payload_Mass_by_NASA(CRS)" 
from SPACEXTBL 
where Customer = "NASA (CRS)"

Running query in 'sqlite:///my_data1.db'

Total_Payload_Mass_by_NASA(CRS)
45596


### Average payload for Falcon 9 v1.1

Average payload mass across Falcon 9 v1.1 launches.

In [48]:
%%sql 
select avg(PAYLOAD_MASS__KG_) 
from SPACEXTBL 
where Booster_Version = "F9 v1.1"

Running query in 'sqlite:///my_data1.db'

avg(PAYLOAD_MASS__KG_)
2928.4


### First successful ground-pad landing

The earliest date on which a ground-pad landing succeeded.

In [49]:
%%sql 
select min(Date) 
from SPACEXTBL 
where Landing_Outcome = "Success (ground pad)"

Running query in 'sqlite:///my_data1.db'

min(Date)
2015-12-22


### Drone-ship successes with medium payload

List boosters that succeeded on a drone ship with a payload between 4,000 and 6,000 kg.

In [50]:
%%sql 
select Booster_Version 
from SPACEXTBL 
where 
Landing_Outcome = "Success (drone ship)" 
and PAYLOAD_MASS__KG_ between 4000 and 6000

Running query in 'sqlite:///my_data1.db'

Booster_Version
F9 FT B1022
F9 FT B1026
F9 FT B1021.2
F9 FT B1031.2


### Successes and failures by mission outcome

Count launches for each mission outcome.

In [51]:
%%sql 
select 
Mission_Outcome, count(*) 
from SPACEXTBL 
group by Mission_Outcome

Running query in 'sqlite:///my_data1.db'

Mission_Outcome,count(*)
Failure (in flight),1
Success,98
Success,1
Success (payload status unclear),1


### Boosters that carried the maximum payload

Find the booster version(s) that carried the heaviest payload, using a subquery.

In [52]:
%config SqlMagic


SqlMagic(Magics, Configurable) options
------------------------------------
SqlMagic.autocommit=<Bool>
    Set autocommit mode
    Current: True
SqlMagic.autolimit=<Int>
    Automatically limit the size of the returned result sets
    Current: None
SqlMagic.autopandas=<Bool>
    Return Pandas DataFrames instead of regular result sets
    Current: False
SqlMagic.autopolars=<Bool>
    Return Polars DataFrames instead of regular result sets
    Current: False
SqlMagic.column_local_vars=<Bool>
    Return data into local variables from column names
    Current: False
SqlMagic.displaycon=<Bool>
    Show connection string after execution
    Current: True
SqlMagic.displaylimit=<Int>
    Automatically limit the number of rows displayed (full result set is still
    stored)
    Current: 0
SqlMagic.dsn_filename=<Unicode>
    Path to DSN file. When the first argument is of the form [section], a
    sqlalchemy connection string is formed from the matching section in the DSN
    file.
    Current: 

In [53]:
%%sql  
select 
Booster_Version, PAYLOAD_MASS__KG_ 
from SPACEXTBL 
where PAYLOAD_MASS__KG_ = (
    select max(PAYLOAD_MASS__KG_) 
    from SPACEXTBL)

Running query in 'sqlite:///my_data1.db'

Booster_Version,PAYLOAD_MASS__KG_
F9 B5 B1048.4,15600
F9 B5 B1049.4,15600
F9 B5 B1051.3,15600
F9 B5 B1056.4,15600
F9 B5 B1048.5,15600
F9 B5 B1051.4,15600
F9 B5 B1049.5,15600
F9 B5 B1060.2,15600
F9 B5 B1058.3,15600
F9 B5 B1051.6,15600


### 2015 drone-ship failures

Show the month, landing outcome, booster version and site for 2015 drone-ship failures.

In [54]:
%%sql 
select 
substr(Date,6,2) as Month ,
Landing_Outcome, 
Booster_Version, Launch_Site  
from SPACEXTBL where substr(Date,0,5) = '2015' 
and Landing_Outcome = 'Failure (drone ship)'

Running query in 'sqlite:///my_data1.db'

Month,Landing_Outcome,Booster_Version,Launch_Site
01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Ranking landing outcomes by count

Count landing outcomes between 2010-06-04 and 2017-03-20, ranked descending.

In [55]:
%%sql 
select Landing_Outcome, count(Landing_Outcome)  
from SPACEXTBL 
where date between "2010-06-04" and "2017-03-20" 
group by Landing_Outcome 
order by count(Landing_Outcome) desc

Running query in 'sqlite:///my_data1.db'

Landing_Outcome,count(Landing_Outcome)
No attempt,10
Success (drone ship),5
Failure (drone ship),5
Success (ground pad),3
Controlled (ocean),3
Uncontrolled (ocean),2
Failure (parachute),2
Precluded (drone ship),1
